In [1]:
import numpy as numpy
import matplotlib.pyplot as plt
import os
import cv2

root_dir = '/Users/shashwatraj/Garud-Plant-Disease-Detection/plantsDataset/plants_dataset'

train_dir = os.path.join(root_dir, 'train')
valid_dir = os.path.join(root_dir, 'valid')

IMG_SIZE = (256,256)


In [2]:
import tqdm

def create_training_data(dir):
    label_list = []
    training_data = []
    for diseases in os.listdir(dir):
        label_list.append(diseases)
        class_num = label_list.index(diseases)
        for img in tqdm.tqdm(os.listdir(os.path.join(dir, diseases))):
            try:
                image = cv2.imread(os.path.join(dir, diseases, img))
                image = cv2.resize(image, IMG_SIZE)
                training_data.append([image, class_num])
            except Exception as e:
                pass
    
    return training_data


In [ ]:
train_data = create_training_data(train_dir)
valid_data = create_training_data(valid_dir)

 55%|█████▍    | 1093/1988 [00:00<00:00, 2275.59it/s]

In [4]:
import random

random.shuffle(train_data)

print(len(train_data))
for sample in train_data[:10]:
    print(sample[1])

70295
19
16
17
16
37
5
17
36
13
1


In [5]:
X_train = []
y_train = []

for features_train, label_train in train_data:
    X_train.append(features_train)
    y_train.append(label_train)

#X_train = numpy.array(X_train).reshape(-1, IMG_SIZE[0], IMG_SIZE[1], 3)

X_valid = []
y_valid = []

for features_valid, label_valid in valid_data:
    X_valid.append(features_valid)
    y_valid.append(label_valid)

#X_valid = numpy.array(X_valid).reshape(-1, IMG_SIZE[0], IMG_SIZE[1], 3)

: 

In [ ]:
import tensorflow as tf

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)) \
                         .batch(16)
val_ds   = tf.data.Dataset.from_tensor_slices((X_valid, y_valid)) \
                         .batch(16)


In [ ]:
'''import numpy as np
# assuming your list-of-3D-arrays is called X_train
X_train = np.stack(X_train).astype("float32")    # now shape is (n, H, W, 3)
X_valid = np.stack(X_valid).astype("float32")
# labels:
y_train = np.array(y_train).astype("uint8")      # shape (n,)
y_valid = np.array(y_valid).astype("uint8")'''




In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Activation, Rescaling, BatchNormalization

model = Sequential([

    # 2) Conv block 1
    Conv2D(64, 3, padding='same', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    Activation('relu'),
    MaxPooling2D(),

    # 3) Conv block 2
    Conv2D(64, 3, padding='same'),
    Activation('relu'),
    MaxPooling2D(),
    Dropout(0.25),

    # 4) Conv block 3
    Conv2D(128, 3, padding='same'),
    Activation('relu'),
    MaxPooling2D(),

    # 5) Conv block 4
    Conv2D(128, 3, padding='same'),
    Activation('relu'),
    MaxPooling2D(),
    Dropout(0.4),

    # 6) Conv block 5
    Conv2D(256, 3, padding='same'),
    Activation('relu'),
    MaxPooling2D(),

    # 7) Classifier head
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(38, activation='linear'),
])

model.summary()

/Users/shashwatraj/garudenv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 256, 256, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64, 64, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     4,194,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         9,766 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,759,654 (18.16 MB)

 Trainable params: 4,759,654 (18.16 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy, CategoricalCrossentropy

model.compile(optimizer='adam', loss = SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
garud = model.fit(
    train_ds, 
    epochs=10,
    validation_data=val_ds,
)